In [0]:
WITH furbooks_used_coupon AS (
    SELECT 
        accountable_entity_id, 
        accountable_entity_type, 
        start_date,
        discount.amount::DECIMAL(10,2) AS discount_amount,
        discount.catalogReferenceId AS coupon_reference_id,
        discount.code AS codes,
        monetary_components_discounts
    FROM furlenco_silver.furbooks_evolve.revenue_recognitions
    LATERAL VIEW explode(from_json(CAST(monetary_components_discounts AS STRING), 'array<struct<amount:double,catalogReferenceId:string,code:string>>')) AS discount
    WHERE vertical = 'FURLENCO_RENTAL'
        AND state NOT IN ('CANCELLED', 'INVALIDATED')
        AND accountable_entity_type IN ('ITEM', 'ATTACHMENT')
        AND start_date > '2025-04-01'
),

final_discount_val AS (
    SELECT 
        fc.*, 
        d.category, 
        d.type AS coupon_type, 
        d.unit AS discount_unit,
        d.exclusive_flows, 
        d.exclusive_platforms, 
        d.exclusive_user_segments
    FROM furbooks_used_coupon AS fc 
    LEFT JOIN furlenco_silver.godfather_evolve.discounts AS d 
        ON fc.coupon_reference_id = d.id
),

-- Sum all discounts per entity per month BEFORE joining base_price
discount_aggregated AS (
    SELECT
        accountable_entity_id,
        accountable_entity_type,
        date_trunc('month', start_date) AS start_month,
        SUM(discount_amount) AS total_discount
    FROM final_discount_val
    GROUP BY 1, 2, 3
),

-- Now join base_price once per entity — no duplication risk
base AS (
    SELECT
        da.*, coalesce('ITEM'||i.id, 'ATTACH'||att.id) as entity_id,
        COALESCE(i.user_id, att.user_id) AS user_id,
        COALESCE(
            get_json_object(CAST(i.pricing_details AS STRING), '$.basePrice'),
            get_json_object(CAST(att.pricing_details AS STRING), '$.basePrice')
        ) AS base_price
    FROM discount_aggregated AS da
    LEFT JOIN furlenco_silver.order_management_systems_evolve.items AS i
        ON i.id = da.accountable_entity_id AND da.accountable_entity_type = 'ITEM'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.attachments AS att
        ON att.id = da.accountable_entity_id AND da.accountable_entity_type = 'ATTACHMENT'
)

SELECT
    start_month,
    SUM(total_discount)  AS total_discount,
    SUM(base_price)      AS base_price_item_attach,
    COUNT(entity_id) as entity,
    COUNT(distinct entity_id) as uni_entity,
    COUNT(distinct user_id) as user_id
FROM base
GROUP BY 1
ORDER BY 1